In [7]:
import os, sys
sys.path.append(os.path.abspath("../../.."))

import pandas as pd
import joblib
import pandas_market_calendars as mcal

from pandas.errors import EmptyDataError
from classes.trading.actionPredictionTrading import ActionPredictionTrading
from classes.neural_networks.architectures.arima_model import ArimaModel

In [2]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

In [3]:
# --- Helpers ---
b3_cal = mcal.get_calendar('B3')

def load_full_series(csv_path: str, stock: str) -> pd.Series:
    """Carrega série completa, indexada só em pregões da B3."""
    df = pd.read_csv(csv_path, parse_dates=['Date'], index_col='Date')
    df.sort_index(inplace=True)
    sched = b3_cal.schedule(start_date=df.index.min(), end_date=df.index.max())
    idx   = sched.index
    return df[stock].reindex(idx).dropna()

def rolling_forecast(model_fit, series: pd.Series, step_size: int = 1) -> pd.Series:
    """Forecast roll-forward estendendo `model_fit` com valores reais."""
    preds = []
    pos = 0
    while pos < len(series):
        h = min(step_size, len(series) - pos)
        yhat = model_fit.forecast(steps=h)
        preds.extend(yhat)
        if pos + h < len(series):
            model_fit = model_fit.extend(series.iloc[pos:pos+h].values)
        pos += h
    return pd.Series(preds, index=series.index)

In [4]:
# 

def print_detailed_performance(strategy_name: str, result_dict: dict, initial_capital: float = 100000.0):
    """
    Pega o dicionário da simulação e imprime um resumo detalhado e formatado.
    """
    # Extrai e calcula as métricas pedidas
    total_trades = result_dict.get('total_trades', 0)
    if total_trades == 0:
        print(f"\n--- Análise de Performance para '{strategy_name}' ---")
        print("    - Nenhuma operação foi realizada.")
        print("-" * 50)
        return

    hit_rate = result_dict['hit_rate']
    final_capital = result_dict['final_capital']
    max_drawdown = result_dict['max_drawdown']

    dias_de_lucro = int(total_trades * hit_rate)
    dias_de_prejuizo = total_trades - dias_de_lucro
    lucro_total_rs = final_capital - initial_capital

    print(f"\n--- Análise de Performance para '{strategy_name}' ---")
    
    print("\n  Resultado Financeiro:")
    print(f"    - Lucro/Prejuízo Total: R$ {lucro_total_rs:,.2f}")
    print(f"    - Capital Final:        R$ {final_capital:,.2f}")

    print("\n  Consistência da Estratégia:")
    print(f"    - Dias de Lucro:        {dias_de_lucro}")
    print(f"    - Dias de Prejuízo:     {dias_de_prejuizo}")
    print(f"    - Total de Trades:      {total_trades}")
    print(f"    - Taxa de Acerto (Hit Rate): {hit_rate:.2%}")

    print("\n  Análise de Risco:")
    print(f"    - Máximo Drawdown:      {max_drawdown:.2%}")
    print("-" * 50)


In [25]:
# --- Parâmetros ---
csv_path   = "../../datasets/b3_dados/processed/assets_concat.csv"
stocks     =  [

    "BBAS3",
]

periods    = {
    "pre_pandemia":     ("2019-01-01", "2019-12-31"),
    "durante_pandemia": ("2020-01-01", "2021-08-31"),
    "pos_pandemia":     ("2021-09-01", "2022-09-30"),
}
arima_dir  = "../../saved_models/arima"
arima_ver  = "1.0"
shares     = 100

In [26]:
# --- Backtest sem retrain ---
results = {}

for stock in stocks:
    print(f"\nAnalyzing stock: {stock}")
    # carrega ARIMA treinado
    arima_path = os.path.join(arima_dir, f"{stock}_arima_v{arima_ver}.pkl")
    arima: ArimaModel = joblib.load(arima_path)

    # carrega série completa
    series = load_full_series(csv_path, stock)

    for period_name, (start, end) in periods.items():
        print(f"\nProcessing period: {period_name} ({start} to {end})")
        subset = series[start:end]
        preds  = rolling_forecast(arima.model_fit, subset, step_size=1)

        # calcula métricas  
        mse = mean_squared_error(subset, preds)
        mae = mean_absolute_error(subset, preds)    
        rmse = np.sqrt(mse)
        r2 = r2_score(subset, preds)

        arima_metrics = {
            'mse': mse,
            'rmse': rmse,
            'mae': mae,
            'r2': r2
        }

        # monta DataFrame para o simulador
        df_bt = pd.DataFrame({
            'Date':      subset.index,
            'actual':    subset.values,
            'predicted': preds.values
        })

        # instancia para backtest
        ap = ActionPredictionTrading(df_bt, ticker='actual', window=1, model_path=None)
        # injeta a coluna de previsões
        ap.df['predicted'] = df_bt['predicted'].reset_index(drop=True)
        ap.df['actual_next'] = ap.df['actual'].shift(-1)
        # roda as simulações
        result_no_stop   = ap.simulate_trading(stop_loss=False, shares_per_trade=shares)
        result_with_stop = ap.simulate_trading(stop_loss=True,  shares_per_trade=shares)
        bh      = ap.simulate_buy_and_hold(shares=shares)

        print_detailed_performance("Estratégia SEM Stop Loss", result_no_stop)
        print_detailed_performance("Estratégia COM Stop Loss", result_with_stop)


        results[(stock, period_name)] = {
            'arima_metrics': arima_metrics,
            'arima_no_stop':   result_no_stop,
            'arima_with_stop': result_with_stop,
            'arima_bh':        bh
        }


Analyzing stock: BBAS3

Processing period: pre_pandemia (2019-01-01 to 2019-12-31)


c:\Users\mathi\.conda\envs\tcc_project\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
c:\Users\mathi\.conda\envs\tcc_project\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(
c:\Users\mathi\.conda\envs\tcc_project\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(



--- Análise de Performance para 'Estratégia SEM Stop Loss' ---

  Resultado Financeiro:
    - Lucro/Prejuízo Total: R$ 513.23
    - Capital Final:        R$ 100,513.23

  Consistência da Estratégia:
    - Dias de Lucro:        128
    - Dias de Prejuízo:     119
    - Total de Trades:      247
    - Taxa de Acerto (Hit Rate): 51.82%

  Análise de Risco:
    - Máximo Drawdown:      0.33%
--------------------------------------------------

--- Análise de Performance para 'Estratégia COM Stop Loss' ---

  Resultado Financeiro:
    - Lucro/Prejuízo Total: R$ 681.39
    - Capital Final:        R$ 100,681.39

  Consistência da Estratégia:
    - Dias de Lucro:        128
    - Dias de Prejuízo:     119
    - Total de Trades:      247
    - Taxa de Acerto (Hit Rate): 51.82%

  Análise de Risco:
    - Máximo Drawdown:      0.28%
--------------------------------------------------

Processing period: durante_pandemia (2020-01-01 to 2021-08-31)


c:\Users\mathi\.conda\envs\tcc_project\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
c:\Users\mathi\.conda\envs\tcc_project\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(
c:\Users\mathi\.conda\envs\tcc_project\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(



--- Análise de Performance para 'Estratégia SEM Stop Loss' ---

  Resultado Financeiro:
    - Lucro/Prejuízo Total: R$ 430.67
    - Capital Final:        R$ 100,430.67

  Consistência da Estratégia:
    - Dias de Lucro:        206
    - Dias de Prejuízo:     206
    - Total de Trades:      412
    - Taxa de Acerto (Hit Rate): 50.00%

  Análise de Risco:
    - Máximo Drawdown:      0.67%
--------------------------------------------------

--- Análise de Performance para 'Estratégia COM Stop Loss' ---

  Resultado Financeiro:
    - Lucro/Prejuízo Total: R$ 1,515.68
    - Capital Final:        R$ 101,515.68

  Consistência da Estratégia:
    - Dias de Lucro:        206
    - Dias de Prejuízo:     206
    - Total de Trades:      412
    - Taxa de Acerto (Hit Rate): 50.00%

  Análise de Risco:
    - Máximo Drawdown:      0.42%
--------------------------------------------------

Processing period: pos_pandemia (2021-09-01 to 2022-09-30)


c:\Users\mathi\.conda\envs\tcc_project\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
c:\Users\mathi\.conda\envs\tcc_project\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(
c:\Users\mathi\.conda\envs\tcc_project\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(



--- Análise de Performance para 'Estratégia SEM Stop Loss' ---

  Resultado Financeiro:
    - Lucro/Prejuízo Total: R$ -35.55
    - Capital Final:        R$ 99,964.45

  Consistência da Estratégia:
    - Dias de Lucro:        134
    - Dias de Prejuízo:     135
    - Total de Trades:      269
    - Taxa de Acerto (Hit Rate): 49.81%

  Análise de Risco:
    - Máximo Drawdown:      0.76%
--------------------------------------------------

--- Análise de Performance para 'Estratégia COM Stop Loss' ---

  Resultado Financeiro:
    - Lucro/Prejuízo Total: R$ 245.82
    - Capital Final:        R$ 100,245.82

  Consistência da Estratégia:
    - Dias de Lucro:        134
    - Dias de Prejuízo:     135
    - Total de Trades:      269
    - Taxa de Acerto (Hit Rate): 49.81%

  Análise de Risco:
    - Máximo Drawdown:      0.57%
--------------------------------------------------


In [27]:

# --- flatten dos resultados em linhas de tabela ---
flat = []
for (stock, period_name), vals in results.items():
    flat.append({
        'stock':           stock,
        'period':          period_name,
        'modelo':          'ARIMA', 
        # 'mse':             vals['arima_metrics']['mse'],
        'rmse':            vals['arima_metrics']['rmse'],
        'mae':             vals['arima_metrics']['mae'],
        'r2':              vals['arima_metrics']['r2'], 

        'total_return_no_stop':   vals['arima_no_stop']['total_return'],
        'hit_rate_no_stop':    vals['arima_no_stop']['hit_rate'],
        'sharpe_no_stop':    vals['arima_no_stop']['sharpe_ratio'],
        'drawdown_no_stop':  vals['arima_no_stop']['max_drawdown'],
        'final_capital_no_stop':   vals['arima_no_stop']['final_capital'],
        'num_trades_no_stop':   vals['arima_no_stop']['total_trades'],
        'profitable_trades_no_stop':   vals['arima_no_stop']['profitable_trades'],
        'unprofitable_trades_no_stop': vals['arima_no_stop']['unprofitable_trades'],
        'avg_return_per_trade_no_stop': vals['arima_no_stop']['avg_return_per_trade'],

        'total_return_stop':      vals['arima_with_stop']['total_return'],
        'hit_rate_stop':       vals['arima_with_stop']['hit_rate'],
        'sharpe_stop':       vals['arima_with_stop']['sharpe_ratio'],
        'drawdown_stop':     vals['arima_with_stop']['max_drawdown'],
        'final_capital_stop':      vals['arima_with_stop']['final_capital'],
        'num_trades_stop':      vals['arima_with_stop']['total_trades'],
        'profitable_trades_stop':   vals['arima_with_stop']['profitable_trades'],
        'unprofitable_trades_stop': vals['arima_with_stop']['unprofitable_trades'],
        'avg_return_per_trade_stop': vals['arima_with_stop']['avg_return_per_trade'],

        # 'retorno_bh':      vals['arima_bh']['total_return'],
        # 'capital_bh':      vals['arima_bh']['final_capital'],
        # 'dias_bh':         vals['arima_bh']['days_held']
    })

df_results = pd.DataFrame(flat)

# --- caminho onde guardar ---
csv_path = "../../datasets/trading/arima_trading_results2.csv"
os.makedirs(os.path.dirname(csv_path), exist_ok=True)

# --- concat incremental ---
if os.path.exists(csv_path):
    try:
        df_prev = pd.read_csv(csv_path)
    except EmptyDataError:
        # arquivo existe, mas vazio: considera DataFrame vazio
        df_prev = pd.DataFrame()
    df_comb = pd.concat([df_prev, df_results], ignore_index=True)
    df_comb.drop_duplicates(subset=['stock','period','modelo'], keep='last', inplace=True)
    df_comb.to_csv(csv_path, index=False)
    print(f"Resultados ARIMA atualizados em {csv_path}")
else:
    # não existia: cria do zero
    df_results.to_csv(csv_path, index=False)
    print(f"Arquivo ARIMA criado: {csv_path}")

# carrega para visualizar
trading_results = pd.read_csv(csv_path)
trading_results



Resultados ARIMA atualizados em ../../datasets/trading/arima_trading_results2.csv


,stock,period,modelo,rmse,mae,r2,total_return_no_stop,hit_rate_no_stop,sharpe_no_stop,drawdown_no_stop,...,avg_return_per_trade_no_stop,total_return_stop,hit_rate_stop,sharpe_stop,drawdown_stop,final_capital_stop,num_trades_stop,profitable_trades_stop,unprofitable_trades_stop,avg_return_per_trade_stop
0,ABEV3,pre_pandemia,ARIMA,0.309300,0.174520,0.806252,0.003610,0.506073,0.064930,0.001785,...,1.461353e-05,0.004326,0.506073,0.080513,0.001612,100432.554798,247,125.0,122.0,1.751234e-05
1,ABEV3,durante_pandemia,ARIMA,0.279217,0.207738,0.974674,0.007017,0.521845,0.061118,0.005268,...,1.703087e-05,0.013802,0.521845,0.141009,0.003950,101380.226535,412,215.0,197.0,3.350064e-05
2,ABEV3,pos_pandemia,ARIMA,0.227804,0.150450,0.895884,-0.001393,0.464684,-0.025855,0.004787,...,-5.179753e-06,-0.000086,0.464684,-0.001742,0.003762,99991.371256,269,125.0,144.0,-3.207712e-07
3,GE,pre_pandemia,ARIMA,2.892162,0.992250,0.581084,-0.012021,0.489627,-0.042201,0.022688,...,-4.988116e-05,0.010425,0.489627,0.045058,0.007883,101042.458549,241,118.0,123.0,4.325554e-05
4,GE,durante_pandemia,ARIMA,1.799579,1.140114,0.983726,0.003249,0.525926,0.005414,0.026899,...,8.022544e-06,0.062088,0.525926,0.127612,0.009926,106208.770061,405,213.0,192.0,1.533030e-04
5,GE,pos_pandemia,ARIMA,1.485102,0.910358,0.969919,0.004433,0.501916,0.014792,0.018332,...,1.698619e-05,0.016078,0.501916,0.057772,0.009872,101607.828606,261,131.0,130.0,6.160263e-05
6,VALE3,pre_pandemia,ARIMA,0.942265,0.493256,0.746741,0.026670,0.530364,0.136941,0.006336,...,1.079774e-04,0.030671,0.530364,0.163619,0.005392,103067.074463,247,131.0,116.0,1.241731e-04
7,VALE3,durante_pandemia,ARIMA,1.375285,0.967235,0.994736,0.042392,0.555825,0.082216,0.016573,...,1.028928e-04,0.072041,0.555825,0.157573,0.011155,107204.132259,412,229.0,183.0,1.748576e-04
8,VALE3,pos_pandemia,ARIMA,3.197419,1.347844,0.807222,-0.029420,0.472119,-0.071678,0.047486,...,-1.093687e-04,-0.004477,0.472119,-0.012372,0.026872,99552.285870,269,127.0,142.0,-1.664365e-05
9,PETR3,pre_pandemia,ARIMA,0.266295,0.146314,0.729418,0.000377,0.449393,0.008720,0.002795,...,1.525643e-06,0.001342,0.449393,0.033189,0.002157,100134.158752,247,111.0,136.0,5.431528e-06


In [ ]:
trading_results["stock"].unique(), trading_results["period"].unique(), trading_results["modelo"].unique()